## Install Dependencies


In [1]:
# Dependencis for the notebook
%pip install spacy pandas torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy
# optuna and optional dependencies
%pip install optuna scikit-learn plotly 
# Qwen optimization (optional) dependencies
%pip install flash-linear-attention causal-conv1d
# Mistral dependencies
%pip install accelerate


/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
import matplotlib
import optuna.visualization
import optuna.importance
import json
from pathlib import Path
from collections import OrderedDict
import modules.llms as llm_parsers
import modules.deepparse_parser as deepparse
import modules.libpostal_client as libpostal_client
import modules.token_classifiers as token_classifiers
from modules.utils import compare_preds, format_time, natural_casing
from typing import NamedTuple
from tqdm.auto import tqdm

from typing import Callable

import abc
import contextlib
import pandas as pd
import time
import pprint
import textwrap
import numpy as np
import modules.train_ner as train_ner
import re

# Total number of addresses in the complete BZK data for calculating estimated time of processing
TOTAL_ADDRESSES = 4_394_539

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

ALL_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "District",
    "State",
    "Region",
    "Country"
]

BZK_ADDRESS_FIELDS = [
    'ApplicantCurrentAddress',
    'VictimBirthPlace',
    'VictimCurrentAddress',     
    'ApplicantBirthPlace',
    'VictimDeathPlace'
]

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## Set hardware accelerator


In [3]:
# List available hardware accelerators for PyTorch
import torch
available_accelerators = []
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
        available_accelerators.append(torch.device(f'cuda:{i}'))
elif torch.accelerator.is_available(): # Support other hardware accelators
    available_accelerators.append(torch.accelerator.current_accelerator())
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')

CUDA - available devices:
  0: NVIDIA A100 80GB PCIe


/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [4]:
if available_accelerators:
    device = available_accelerators[0]
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+cu128, Device: cuda:0


## Load Dataset

Read all splits and concat them

In [5]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_merged : pd.DataFrame = pd.concat(
    [bzkopen_train, bzkopen_val, bzkopen_test], 
    keys=['train', 'val', 'test'], 
    names=['split', 'row']
)
display(bzkopen_merged.sample(5))

field  \
split row                            
val   115  ApplicantCurrentAddress   
train 579         VictimBirthPlace   
      657      ApplicantBirthPlace   
      523  ApplicantCurrentAddress   
test  37   ApplicantCurrentAddress   

                                                 FullAddress UnitNumber  \
split row                                                                 
val   115   Bielefeld, Schildescherstr. 88 Reg. Bez. Detmold        NaN   
train 579                                             Mering        NaN   
      657                                           Mannheim        NaN   
      523                             Göttingen Jüdenstr. 45        NaN   
test  37   Nürnberg, Vestnertorgraben 13/Kreis Mittelfranken        NaN   

          HouseNumber        StreetName Neighborhood       City  \
split row                                                         
val   115          88  Schildescherstr.          NaN  Bielefeld   
train 579         NaN               NaN          NaN     Mering   
      657         NaN               NaN          NaN   Mannheim   
      523          45         Jüdenstr.          NaN  Göttingen   
test  37           13  Vestnertorgraben          NaN   Nürnberg   

                District   Region State Country PostalCode  
split row                                                   
val   115            NaN  Detmold   NaN     NaN        NaN  
train 579            NaN      NaN   NaN     NaN        NaN  
      657            NaN      NaN   NaN     NaN        NaN  
      523            NaN      NaN   NaN     NaN        NaN  
test  37   Mittelfranken      NaN   NaN     NaN        NaN

## Create folds



In [6]:
N_FOLDS = 10

# Set shuffle seed for reproducibility
SEED=4841762


In [7]:
# Define paths for saving models and predictions
models_path = Path("models") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
models_path.mkdir(parents=True, exist_ok=True)
preds_path = Path("experiments_data") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
preds_path.mkdir(parents=True, exist_ok=True)

In [8]:
class Fold(NamedTuple):
    fold_idx: int
    train_data: pd.DataFrame
    val_data: pd.DataFrame
    models_path: Path
    preds_path: Path

In [9]:

shuffled_bzkopen = bzkopen_merged.sample(frac=1, random_state=SEED)

fold_indices = np.array_split(shuffled_bzkopen.index, N_FOLDS)

folds = []

for fold_idx, val_indices in enumerate(fold_indices):
    train_indices = shuffled_bzkopen.index.difference(val_indices)
    fold_models_path = models_path / f"fold_{fold_idx}"
    fold_models_path.mkdir(parents=True, exist_ok=True)
    fold_preds_path = preds_path / f"fold_{fold_idx}"
    fold_preds_path.mkdir(parents=True, exist_ok=True)
    folds.append(Fold(
        fold_idx, 
        shuffled_bzkopen.loc[train_indices], 
        shuffled_bzkopen.loc[val_indices],
        fold_models_path, 
        fold_preds_path
    ))
display(pd.DataFrame(
    [[len(x) for x in [fold.val_data, fold.train_data]] for fold in folds], 
    columns=['val_fold_size', 'train_fold_size'])
)

,val_fold_size,train_fold_size
0,108,967
1,108,967
2,108,967
3,108,967
4,108,967
5,107,968
6,107,968
7,107,968
8,107,968
9,107,968


# Train Spacy NER models

These models are used for the pattern example matching strategy so they will be reused over multiple different experiments. Therefore it is useful to train them in advance.

In [10]:
for fold in tqdm(folds, unit="fold"):
    output_dir = fold.models_path / 'ner_bzk'
    if output_dir.exists():
        print(f"Fold {fold.fold_idx} already has a trained model, skipping training.")
        continue
    train_ner.train(
        n_iter=30,
        output_dir=output_dir,
        train_df=fold.train_data,
        val_df=fold.val_data
    )

  0%|          | 0/10 [00:00<?, ?fold/s]

Fold 0 already has a trained model, skipping training.
Fold 1 already has a trained model, skipping training.
Fold 2 already has a trained model, skipping training.
Fold 3 already has a trained model, skipping training.
Fold 4 already has a trained model, skipping training.
Fold 5 already has a trained model, skipping training.
Fold 6 already has a trained model, skipping training.
Fold 7 already has a trained model, skipping training.
Fold 8 already has a trained model, skipping training.
Fold 9 already has a trained model, skipping training.


In [11]:
# Metrics on the required entities (Country, City, StreetName, HouseNumber)
required_results = OrderedDict()
specific_results = OrderedDict()


class FoldEvaluator:
    def __init__(self, model_name : str, supported_entities : list[str]):
        # Metrics on the required entities (Country, City, StreetName, HouseNumber)
        self._required_metric_records = None
        self._specific_metric_records = None
        self.required_metrics = None
        self.specific_metrics = None
        self.model_name = model_name
        self.supported_entities = supported_entities
    
    def __enter__(self):
        self.required_metrics = None
        self.specific_metrics = None
        self._required_metric_records = []
        self._specific_metric_records = []
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        global required_results, specific_results
        self.required_metrics = pd.DataFrame(self._required_metric_records).aggregate(['mean', 'std']).unstack()
        self.specific_metrics = pd.DataFrame(self._specific_metric_records).aggregate(['mean', 'std']).unstack()
        required_results[self.model_name] = self.required_metrics
        specific_results[self.model_name] = self.specific_metrics
        print(f"{self.model_name} - Metrics for Country City Street House:")
        display(self.required_metrics)
        print(f"{self.model_name} - Specific Metrics:")
        display(self.specific_metrics.unstack(level=0))
        self._required_metric_records = None
        self._specific_metric_records = None

    def _record_predictions(self, fold : Fold, preds : list[dict]):
        preds_df = pd.DataFrame(preds)
        fold_metrics = compare_preds(preds_df, fold.val_data, target_columns=REQUIRED_ENTITIES)
        self._required_metric_records.append(pd.Series(fold_metrics))
        specific_fold_metrics = OrderedDict()
        for entity in self.supported_entities:
            entity_metrics = compare_preds(preds_df, fold.val_data, target_columns=[entity])
            specific_fold_metrics[entity] = pd.Series(entity_metrics)
        for bzk_field in BZK_ADDRESS_FIELDS:
            mask = fold.val_data['field'] == bzk_field
            field_metrics = compare_preds(preds_df[mask.reset_index(drop=True)], fold.val_data[mask], target_columns=REQUIRED_ENTITIES)
            specific_fold_metrics[bzk_field] = pd.Series(field_metrics)
        self._specific_metric_records.append(pd.concat(specific_fold_metrics, names=['Field/Entity', 'Metric']))

    def load_preds_and_skip(self, folds : list[Fold]) -> list[Fold]:
        """
        For each fold, load its predictions if available on disk.
        Returns the folds for which predictions were not loaded
        """
        unloaded_folds = []
        for fold in folds:
            preds_file = fold.preds_path / self.model_name / "preds.json"
            if preds_file.exists():
                print(f"Fold {fold.fold_idx} already has predictions for {self.model_name}, loading from file.")
                with open(preds_file, "r") as f:
                    preds = json.load(f)
                self._record_predictions(fold, preds)
            else:
                unloaded_folds.append(fold)
        return unloaded_folds
            
    
    def run_on_fold(self, fold : Fold, model):
        assert self._required_metric_records is not None and self._specific_metric_records is not None, "FoldMetricRecorder must be used as a context manager"
        preds_file = fold.preds_path / self.model_name / "preds.json"
        preds_file.parent.mkdir(parents=True, exist_ok=True)
        preds = model.parse_addresses(fold.val_data['FullAddress'].tolist())
        with open(preds_file, "w") as f:
            json.dump(preds, f, indent=4)
        self._record_predictions(fold, preds)
    

In [12]:
with FoldEvaluator("libpostal", libpostal_client.LIBPOSTAL_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = libpostal_client.LibpostalClient()
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        model.close()

Fold 0 already has predictions for libpostal, loading from file.
Fold 1 already has predictions for libpostal, loading from file.
Fold 2 already has predictions for libpostal, loading from file.
Fold 3 already has predictions for libpostal, loading from file.
Fold 4 already has predictions for libpostal, loading from file.
Fold 5 already has predictions for libpostal, loading from file.
Fold 6 already has predictions for libpostal, loading from file.
Fold 7 already has predictions for libpostal, loading from file.
Fold 8 already has predictions for libpostal, loading from file.
Fold 9 already has predictions for libpostal, loading from file.
libpostal - Metrics for Country City Street House:


accuracy                   mean    0.687913
                           std     0.032383
precision                  mean    0.637836
                           std     0.045664
recall                     mean    0.419072
                           std     0.041448
f1                         mean    0.505555
                           std     0.043244
accuracy_with_tol_1        mean    0.698390
                           std     0.033153
accuracy_with_tol_2        mean    0.706533
                           std     0.030905
accuracy_with_tol_3        mean    0.721655
                           std     0.029771
accuracy_with_tol_4        mean    0.742580
                           std     0.032508
average_levenshtein        mean    2.488696
                           std     0.277225
average_similarity         mean    0.728394
                           std     0.028393
average_levenshtein_match  mean    3.200120
                           std     0.437061
average_similarity_match   mean 

libpostal - Specific Metrics:


Field/Entity                    HouseNumber  StreetName       City     State  \
Metric                                                                         
accuracy                  mean     0.869756    0.771218   0.319064  0.948884   
                          std      0.038574    0.045804   0.043952  0.022337   
precision                 mean     0.748952    0.533138   0.615605  0.667500   
                          std      0.045200    0.055681   0.096090  0.244143   
recall                    mean     0.679602    0.524948   0.299415  0.468452   
                          std      0.063833    0.090766   0.047211  0.175110   
f1                        mean     0.712181    0.528305   0.401641  0.536922   
                          std      0.053743    0.072998   0.058985  0.181123   
accuracy_with_tol_1       mean     0.881862    0.774013   0.336743  0.951670   
                          std      0.039570    0.047422   0.043739  0.020808   
accuracy_with_tol_2       mean     0.906057    0.774013   0.337677  0.952596   
                          std      0.025728    0.047422   0.042482  0.021559   
accuracy_with_tol_3       mean     0.923728    0.780512   0.348849  0.955400   
                          std      0.023133    0.047762   0.039812  0.023401   
accuracy_with_tol_4       mean     0.950727    0.793510   0.374879  0.976791   
                          std      0.023992    0.048253   0.043310  0.017032   
average_levenshtein       mean     0.582321    2.770093   5.511111  0.272300   
                          std      0.219950    0.545948   0.370673  0.116759   
average_similarity        mean     0.894377    0.836213   0.382294  0.951692   
                          std      0.032673    0.033275   0.038977  0.021046   
average_levenshtein_match mean     0.616913    3.066767  12.104832  0.288211   
                          std      0.244294    0.665067   1.730728  0.130935   
average_similarity_match  mean     0.940524    0.921863   0.831204  0.999054   
                          std      0.016100    0.020669   0.047889  0.002024   
no_match_rate             mean     0.049308    0.093025   0.539607  0.047404   
                          std      0.020619    0.025561   0.043935  0.021112   

Field/Entity                     Country  PostalCode  ApplicantCurrentAddress  \
Metric                                                                          
accuracy                  mean  0.791615    0.468354                 0.529093   
                          std   0.040104    0.493989                 0.054128   
precision                 mean  0.740418    0.033333                 0.639412   
                          std   0.122392    0.105409                 0.071530   
recall                    mean  0.317047    0.016667                 0.411403   
                          std   0.092088    0.052705                 0.070355   
f1                        mean  0.441058    0.022222                 0.500029   
                          std   0.109171    0.070273                 0.072820   
accuracy_with_tol_1       mean  0.800943    0.468354                 0.540611   
                          std   0.040885    0.493989                 0.053290   
accuracy_with_tol_2       mean  0.808385    0.473001                 0.553091   
                          std   0.043867    0.498847                 0.047712   
accuracy_with_tol_3       mean  0.833532    0.473001                 0.578030   
                          std   0.044589    0.498847                 0.043286   
accuracy_with_tol_4       mean  0.851203    0.475805                 0.615146   
                          std   0.045661    0.501819                 0.045510   
average_levenshtein       mean  1.091260    0.501237                 3.955788   
                          std   0.255998    0.262921                 0.495107   
average_similarity        mean  0.800691    0.468974                 0.592596   
                          std   0.041506    0.494633      

In [13]:
with FoldEvaluator("deepparse", deepparse.DEEPPARSE_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = deepparse.DeepParseParser(device=device)
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for deepparse, loading from file.
Fold 1 already has predictions for deepparse, loading from file.
Fold 2 already has predictions for deepparse, loading from file.
Fold 3 already has predictions for deepparse, loading from file.
Fold 4 already has predictions for deepparse, loading from file.
Fold 5 already has predictions for deepparse, loading from file.
Fold 6 already has predictions for deepparse, loading from file.
Fold 7 already has predictions for deepparse, loading from file.
Fold 8 already has predictions for deepparse, loading from file.
Fold 9 already has predictions for deepparse, loading from file.
deepparse - Metrics for Country City Street House:


accuracy                   mean    0.570431
                           std     0.031744
precision                  mean    0.321375
                           std     0.033958
recall                     mean    0.271941
                           std     0.035722
f1                         mean    0.294381
                           std     0.034086
accuracy_with_tol_1        mean    0.577862
                           std     0.031669
accuracy_with_tol_2        mean    0.593216
                           std     0.026731
accuracy_with_tol_3        mean    0.609724
                           std     0.024234
accuracy_with_tol_4        mean    0.630428
                           std     0.027539
average_levenshtein        mean    3.934809
                           std     0.331924
average_similarity         mean    0.648879
                           std     0.026140
average_levenshtein_match  mean    5.110830
                           std     0.502228
average_similarity_match   mean 

deepparse - Specific Metrics:


Field/Entity                    HouseNumber  StreetName       City   Country  \
Metric                                                                         
accuracy                  mean     0.864183    0.472430   0.253037  0.692073   
                          std      0.037154    0.054841   0.040026  0.041137   
precision                 mean     0.834101    0.048421   0.318456  0.272803   
                          std      0.053190    0.019713   0.056522  0.155691   
recall                    mean     0.656874    0.066416   0.255519  0.079102   
                          std      0.079146    0.026924   0.040036  0.041755   
f1                        mean     0.733792    0.055821   0.283273  0.118521   
                          std      0.065601    0.022270   0.046228  0.056528   
accuracy_with_tol_1       mean     0.886483    0.472430   0.259536  0.692999   
                          std      0.039400    0.054841   0.035509  0.039892   
accuracy_with_tol_2       mean     0.939529    0.474290   0.264183  0.694860   
                          std      0.020732    0.056261   0.035147  0.038122   
accuracy_with_tol_3       mean     0.963699    0.478020   0.271625  0.725554   
                          std      0.016121    0.058212   0.035842  0.039199   
accuracy_with_tol_4       mean     0.976739    0.489174   0.292108  0.763690   
                          std      0.010060    0.060244   0.038645  0.043218   
average_levenshtein       mean     0.453141    6.733506   6.726912  1.825675   
                          std      0.168387    0.790466   0.503971  0.308215   
average_similarity        mean     0.874064    0.618736   0.405451  0.697264   
                          std      0.034185    0.043409   0.030721  0.041245   
average_levenshtein_match mean     0.512898    8.213621  10.050439  2.631671   
                          std      0.199188    1.019882   0.779379  0.570586   
average_similarity_match  mean     0.980704    0.753386   0.605467  0.994171   
                          std      0.010189    0.041216   0.043294  0.003977   
no_match_rate             mean     0.108818    0.178704   0.330270  0.298615   
                          std      0.031054    0.035557   0.021273  0.041969   

Field/Entity                    PostalCode  ApplicantCurrentAddress  \
Metric                                                                
accuracy                  mean    0.922793                 0.352289   
                          std     0.022864                 0.026584   
precision                 mean    0.025000                 0.309011   
                          std     0.079057                 0.054827   
recall                    mean    0.016667                 0.230186   
                          std     0.052705                 0.045794   
f1                        mean    0.020000                 0.263626   
                          std     0.063246                 0.049411   
accuracy_with_tol_1       mean    0.928375                 0.365328   
                          std     0.022018                 0.022965   
accuracy_with_tol_2       mean    0.939521                 0.389361   
                          std     0.020803                 0.022312   
accuracy_with_tol_3       mean    0.948840                 0.419085   
                          std     0.019791                 0.014867   
accuracy_with_tol_4       mean    0.956308                 0.449611   
                          std     0.020090                 0.015847   
average_levenshtein       mean    0.540247                 6.415092   
                          std     0.204010                 0.416029   
average_similarity        mean    0.923041                 0.466432   
                          std     0.022868                 0.027902   
average_levenshtein_match mean    0.588115                 9.573635   
                          std     0.233693                 0.866804   
average_similarity_match  mean    0.997275             

In [14]:
with FoldEvaluator("xlm-roberta-large", token_classifiers.FIGHTING_CRIME_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = token_classifiers.TokenClassifierAddressParser(
            model_name="hm-haitham/xlm-roberta-large-address-parser", 
            device=device,
            aggregation_strategy="average"
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for xlm-roberta-large, loading from file.
Fold 1 already has predictions for xlm-roberta-large, loading from file.
Fold 2 already has predictions for xlm-roberta-large, loading from file.
Fold 3 already has predictions for xlm-roberta-large, loading from file.
Fold 4 already has predictions for xlm-roberta-large, loading from file.
Fold 5 already has predictions for xlm-roberta-large, loading from file.
Fold 6 already has predictions for xlm-roberta-large, loading from file.
Fold 7 already has predictions for xlm-roberta-large, loading from file.
Fold 8 already has predictions for xlm-roberta-large, loading from file.
Fold 9 already has predictions for xlm-roberta-large, loading from file.
xlm-roberta-large - Metrics for Country City Street House:


accuracy                   mean    0.805393
                           std     0.017126
precision                  mean    0.709732
                           std     0.037648
recall                     mean    0.657383
                           std     0.032906
f1                         mean    0.682448
                           std     0.033984
accuracy_with_tol_1        mean    0.816312
                           std     0.014871
accuracy_with_tol_2        mean    0.822359
                           std     0.013141
accuracy_with_tol_3        mean    0.833511
                           std     0.014613
accuracy_with_tol_4        mean    0.848875
                           std     0.016481
average_levenshtein        mean    1.430642
                           std     0.191547
average_similarity         mean    0.840967
                           std     0.012325
average_levenshtein_match  mean    1.589168
                           std     0.215828
average_similarity_match   mean 

xlm-roberta-large - Specific Metrics:


Field/Entity                     Country     State      City  StreetName  \
Metric                                                                     
accuracy                  mean  0.909770  0.925632  0.517272    0.874507   
                          std   0.023154  0.029648  0.034361    0.035855   
precision                 mean  0.799440  0.287460  0.599839    0.746411   
                          std   0.098036  0.214509  0.049758    0.078634   
recall                    mean  0.795060  0.227500  0.520498    0.741961   
                          std   0.074286  0.148682  0.038023    0.088406   
f1                        mean  0.795242  0.250404  0.557059    0.743500   
                          std   0.074995  0.170906  0.041079    0.080808   
accuracy_with_tol_1       mean  0.922785  0.932139  0.519133    0.886587   
                          std   0.020567  0.031562  0.036684    0.030620   
accuracy_with_tol_2       mean  0.927440  0.934000  0.520059    0.886587   
                          std   0.020439  0.032290  0.036783    0.030620   
accuracy_with_tol_3       mean  0.937660  0.947015  0.533091    0.894946   
                          std   0.019177  0.028762  0.035266    0.034307   
accuracy_with_tol_4       mean  0.958143  0.964694  0.550762    0.900537   
                          std   0.014660  0.021348  0.034051    0.033458   
average_levenshtein       mean  0.419514  0.410012  3.989339    1.055114   
                          std   0.114472  0.217234  0.416626    0.360446   
average_similarity        mean  0.916798  0.931931  0.602209    0.906168   
                          std   0.021278  0.031553  0.029204    0.024307   
average_levenshtein_match mean  0.458388  0.441647  5.128576    1.129630   
                          std   0.133956  0.240395  0.613353    0.396491   
average_similarity_match  mean  0.996535  0.989823  0.772910    0.967412   
                          std   0.004156  0.004951  0.032848    0.021041   
no_match_rate             mean  0.080002  0.058550  0.220535    0.063240   
                          std   0.021531  0.029339  0.029331    0.018911   

Field/Entity                    HouseNumber  ApplicantCurrentAddress  \
Metric                                                                 
accuracy                  mean     0.920024                 0.706104   
                          std      0.018625                 0.036906   
precision                 mean     0.844150                 0.717041   
                          std      0.046113                 0.058461   
recall                    mean     0.799598                 0.648604   
                          std      0.040271                 0.050138   
f1                        mean     0.821117                 0.680719   
                          std      0.041545                 0.050893   
accuracy_with_tol_1       mean     0.936743                 0.725455   
                          std      0.012259                 0.034663   
accuracy_with_tol_2       mean     0.955348                 0.732211   
                          std      0.011438                 0.035077   
accuracy_with_tol_3       mean     0.968345                 0.752802   
                          std      0.011017                 0.033414   
accuracy_with_tol_4       mean     0.986059                 0.777638   
                          std      0.007881                 0.036337   
average_levenshtein       mean     0.258602                 2.176361   
                          std      0.059059                 0.337750   
average_similarity        mean     0.938691                 0.758700   
                          std      0.011627                 0.027834   
average_levenshtein_match mean     0.268146                 2.522452   
                          std      0.062575                 0.402124   
average_similarity_match  mean     0.972128                 0.878942   
                          std      0.007887                 0.033107   

In [15]:
with FoldEvaluator("SpaCy", supported_entities=ALL_ENTITIES) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        for fold in tqdm(folds_to_process, unit="fold"):
            model = train_ner.SpaCyAddressParser(model_dir=fold.models_path / 'ner_bzk')
            evaluator.run_on_fold(fold, model)

Fold 0 already has predictions for SpaCy, loading from file.
Fold 1 already has predictions for SpaCy, loading from file.
Fold 2 already has predictions for SpaCy, loading from file.
Fold 3 already has predictions for SpaCy, loading from file.
Fold 4 already has predictions for SpaCy, loading from file.
Fold 5 already has predictions for SpaCy, loading from file.
Fold 6 already has predictions for SpaCy, loading from file.
Fold 7 already has predictions for SpaCy, loading from file.
Fold 8 already has predictions for SpaCy, loading from file.
Fold 9 already has predictions for SpaCy, loading from file.
SpaCy - Metrics for Country City Street House:


accuracy                   mean    0.889339
                           std     0.029299
precision                  mean    0.853173
                           std     0.032473
recall                     mean    0.814881
                           std     0.044588
f1                         mean    0.833505
                           std     0.038487
accuracy_with_tol_1        mean    0.898163
                           std     0.030274
accuracy_with_tol_2        mean    0.906304
                           std     0.027338
accuracy_with_tol_3        mean    0.911193
                           std     0.025994
accuracy_with_tol_4        mean    0.917709
                           std     0.025050
average_levenshtein        mean    0.891310
                           std     0.339198
average_similarity         mean    0.912928
                           std     0.024751
average_levenshtein_match  mean    0.953698
                           std     0.385043
average_similarity_match   mean 

SpaCy - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.902328    0.878202      0.893129   
                          std      0.032101    0.043076      0.038305   
precision                 mean     0.816794    0.807996      0.511732   
                          std      0.075382    0.053107      0.297710   
recall                    mean     0.778129    0.729742      0.332692   
                          std      0.062919    0.082073      0.157522   
f1                        mean     0.796591    0.766370      0.379130   
                          std      0.066641    0.068791      0.179513   
accuracy_with_tol_1       mean     0.924628    0.880054      0.893129   
                          std      0.033647    0.043758      0.038305   
accuracy_with_tol_2       mean     0.950692    0.881906      0.893129   
                          std      0.018680    0.043695      0.038305   
accuracy_with_tol_3       mean     0.958134    0.885618      0.894055   
                          std      0.018275    0.044373      0.037803   
accuracy_with_tol_4       mean     0.964642    0.894929      0.895916   
                          std      0.015765    0.045907      0.036132   
average_levenshtein       mean     0.456983    1.250511      1.102302   
                          std      0.222889    0.636207      0.394527   
average_similarity        mean     0.925193    0.905153      0.895510   
                          std      0.024794    0.036524      0.037790   
average_levenshtein_match mean     0.479478    1.359887      1.241778   
                          std      0.237589    0.745530      0.468732   
average_similarity_match  mean     0.965501    0.970893      0.996440   
                          std      0.014344    0.016226      0.005109   
no_match_rate             mean     0.041866    0.067870      0.101289   
                          std      0.014062    0.029257      0.037580   

Field/Entity                        City  District     State    Region  \
Metric                                                                   
accuracy                  mean  0.815040  0.931187  0.950684  0.000000   
                          std   0.055656  0.030649  0.015281  0.000000   
precision                 mean  0.859895  0.695310  0.736667  0.000000   
                          std   0.049514  0.155584  0.233967  0.000000   
recall                    mean  0.839322  0.719165  0.413333  0.000000   
                          std   0.044755  0.164600  0.271660  0.000000   
f1                        mean  0.849330  0.696681  0.484380  0.000000   
                          std   0.045426  0.139027  0.196587  0.000000   
accuracy_with_tol_1       mean  0.823390  0.931187  0.950684  0.000000   
                          std   0.051925  0.030649  0.015281  0.000000   
accuracy_with_tol_2       mean  0.827103  0.932113  0.954396  0.000000   
                          std   0.051449  0.031228  0.014944  0.000000   
accuracy_with_tol_3       mean  0.830841  0.933048  0.956265  0.000000   
                          std   0.051297  0.031756  0.013977  0.000000   
accuracy_with_tol_4       mean  0.839218  0.940498  0.983255  0.000000   
                          std   0.050801  0.026962  0.013748  0.000000   
average_levenshtein       mean  1.589339  0.568051  0.255858  0.685575   
                          std   0.564617  0.275778  0.105661  0.118518   
average_similarity        mean  0.856521  0.932678  0.951514  0.000000   
                          std   0.047629  0.030816  0.015097  0.000000   
average_levenshtein_match mean  1.777726  0.617256  0.269898  0.000000   
                          std   0.690156  0.314992  0.114956  0.000000   
average_similarity_match  mean  0.945896  0.998622  0.998921  0.000000   
                          std   0.019828  0.002528  0.003411  0.000000   
no_match_rate             mean 

In [16]:
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

prompt0_template = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_0.txt").read_text())

with FoldEvaluator("Qwen3.5-9B-prompt0-0shot", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt0_template
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 1 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Qwen3.5-9B-prompt0-0shot - Metrics for Country City Street House:


accuracy                   mean    0.895360
                           std     0.010901
precision                  mean    0.859612
                           std     0.019023
recall                     mean    0.818855
                           std     0.020621
f1                         mean    0.838674
                           std     0.018257
accuracy_with_tol_1        mean    0.901179
                           std     0.010211
accuracy_with_tol_2        mean    0.906525
                           std     0.010180
accuracy_with_tol_3        mean    0.916295
                           std     0.011246
accuracy_with_tol_4        mean    0.927466
                           std     0.010471
average_levenshtein        mean    0.718092
                           std     0.120217
average_similarity         mean    0.919957
                           std     0.010629
average_levenshtein_match  mean    0.762734
                           std     0.135021
average_similarity_match   mean 

Qwen3.5-9B-prompt0-0shot - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.952605    0.914417      0.893977   
                          std      0.017644    0.027352      0.033616   
precision                 mean     0.879931    0.817927      0.366535   
                          std      0.049646    0.047746      0.152542   
recall                    mean     0.889203    0.819884      0.392033   
                          std      0.043193    0.052617      0.158993   
f1                        mean     0.884422    0.818637      0.376215   
                          std      0.045377    0.048044      0.151125   
accuracy_with_tol_1       mean     0.955400    0.925606      0.895829   
                          std      0.017277    0.020906      0.035464   
accuracy_with_tol_2       mean     0.973053    0.926532      0.905140   
                          std      0.013399    0.022438      0.037289   
accuracy_with_tol_3       mean     0.985125    0.930253      0.906066   
                          std      0.010900    0.024390      0.038222   
accuracy_with_tol_4       mean     0.991641    0.941433      0.908844   
                          std      0.009211    0.022628      0.036821   
average_levenshtein       mean     0.162600    0.710038      0.867792   
                          std      0.082917    0.279172      0.370155   
average_similarity        mean     0.965336    0.947484      0.902583   
                          std      0.015034    0.020372      0.033486   
average_levenshtein_match mean     0.166238    0.737136      0.963919   
                          std      0.086696    0.302984      0.452624   
average_similarity_match  mean     0.981775    0.977451      0.988018   
                          std      0.009093    0.008635      0.006526   
no_match_rate             mean     0.016710    0.030659      0.086483   
                          std      0.014994    0.019025      0.033039   

Field/Entity                        City     State  District   Country  \
Metric                                                                   
accuracy                  mean  0.786068  0.933982  0.866978  0.928349   
                          std   0.026841  0.018727  0.024388  0.021075   
precision                 mean  0.863233  0.490490  0.309574  0.889508   
                          std   0.040392  0.083124  0.116266  0.068140   
recall                    mean  0.796758  0.905278  0.373518  0.792191   
                          std   0.028631  0.105136  0.153772  0.050585   
f1                        mean  0.828443  0.633103  0.332916  0.836703   
                          std   0.031370  0.083489  0.120851  0.047710   
accuracy_with_tol_1       mean  0.789780  0.936777  0.869773  0.933930   
                          std   0.024367  0.018909  0.021838  0.016757   
accuracy_with_tol_2       mean  0.790715  0.939573  0.871634  0.935800   
                          std   0.023534  0.019626  0.023052  0.018901   
accuracy_with_tol_3       mean  0.802821  0.943294  0.875363  0.946980   
                          std   0.025266  0.018174  0.022326  0.017501   
accuracy_with_tol_4       mean  0.813058  0.947015  0.889313  0.963733   
                          std   0.024328  0.019962  0.023295  0.022028   
average_levenshtein       mean  1.636951  0.479751  1.002838  0.362781   
                          std   0.211727  0.176496  0.245957  0.149186   
average_similarity        mean  0.828544  0.936191  0.883041  0.938466   
                          std   0.019573  0.018005  0.028377  0.018148   
average_levenshtein_match mean  1.874901  0.513925  1.122111  0.386760   
                          std   0.261043  0.194630  0.308702  0.165008   
average_similarity_match  mean  0.947629  0.997378  0.979866  0.993953   
                          std   0.012302  0.003196  0.013200  0.005593   
no_match_rate             mean 

In [17]:
prompt_llama_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_llama_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "City", "State", "Country"]
n_examples = 14
embedding_model = "multi-qa-distilbert-cos-v1"
similarity_threshold = 0.05


with FoldEvaluator("Llama-3-8B-model-best", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.LlamaAddressParsingModel(
            model_name="meta-llama/Meta-Llama-3-8B-Instruct", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_llama_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            model.example_strategy = embedding_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del embedding_similarity
        del model

Fold 0 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 1 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 2 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 3 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 4 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 5 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 6 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 7 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 8 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 9 already has predictions for Llama-3-8B-model-best, loading from file.
Llama-3-8B-model-best - Metrics for Country City Street House:


accuracy                   mean    0.938664
                           std     0.018643
precision                  mean    0.894843
                           std     0.030239
recall                     mean    0.918821
                           std     0.022619
f1                         mean    0.906638
                           std     0.026144
accuracy_with_tol_1        mean    0.943077
                           std     0.018285
accuracy_with_tol_2        mean    0.946794
                           std     0.016662
accuracy_with_tol_3        mean    0.951899
                           std     0.014337
accuracy_with_tol_4        mean    0.955393
                           std     0.015989
average_levenshtein        mean    0.446528
                           std     0.149626
average_similarity         mean    0.953507
                           std     0.016208
average_levenshtein_match  mean    0.462607
                           std     0.161507
average_similarity_match   mean 

Llama-3-8B-model-best - Specific Metrics:


Field/Entity                    HouseNumber  StreetName      City     State  \
Metric                                                                        
accuracy                  mean     0.960038    0.950796  0.891243  0.961899   
                          std      0.021431    0.026111  0.028757  0.015978   
precision                 mean     0.907071    0.916950  0.893016  0.667530   
                          std      0.055222    0.042304  0.030414  0.136060   
recall                    mean     0.928186    0.895261  0.915229  0.919167   
                          std      0.048554    0.056008  0.020704  0.125096   
f1                        mean     0.917313    0.905887  0.903895  0.759050   
                          std      0.050136    0.048993  0.024325  0.094442   
accuracy_with_tol_1       mean     0.963768    0.954500  0.895890  0.962825   
                          std      0.022890    0.021929  0.028361  0.016311   
accuracy_with_tol_2       mean     0.976774    0.955426  0.895890  0.964685   
                          std      0.015292    0.022087  0.028361  0.016803   
accuracy_with_tol_3       mean     0.984199    0.960981  0.897750  0.967480   
                          std      0.014516    0.018870  0.029113  0.016486   
accuracy_with_tol_4       mean     0.989780    0.962850  0.899619  0.977691   
                          std      0.011926    0.020426  0.029668  0.014017   
average_levenshtein       mean     0.166277    0.426142  0.937080  0.267740   
                          std      0.109930    0.261028  0.256127  0.148532   
average_similarity        mean     0.968466    0.964485  0.922561  0.962639   
                          std      0.016997    0.019341  0.024990  0.016202   
average_levenshtein_match mean     0.170886    0.441946  0.976860  0.280213   
                          std      0.114110    0.278007  0.281009  0.158185   
average_similarity_match  mean     0.989556    0.991147  0.958039  0.999810   
                          std      0.006944    0.004986  0.012046  0.000602   
no_match_rate             mean     0.021366    0.026921  0.037150  0.037175   
                          std      0.011587    0.017677  0.016837  0.016311   

Field/Entity                     Country  ApplicantCurrentAddress  \
Metric                                                              
accuracy                  mean  0.952579                 0.913501   
                          std   0.026718                 0.037094   
precision                 mean  0.859572                 0.902161   
                          std   0.073692                 0.039761   
recall                    mean  0.955060                 0.915653   
                          std   0.037666                 0.038083   
f1                        mean  0.904052                 0.908786   
                          std   0.055825                 0.037980   
accuracy_with_tol_1       mean  0.958152                 0.919734   
                          std   0.024432                 0.035296   
accuracy_with_tol_2       mean  0.959086                 0.927581   
                          std   0.024404                 0.030737   
accuracy_with_tol_3       mean  0.964668                 0.937811   
                          std   0.019425                 0.022277   
accuracy_with_tol_4       mean  0.969323                 0.943915   
                          std   0.019563                 0.023295   
average_levenshtein       mean  0.256611                 0.642060   
                          std   0.135530                 0.263531   
average_similarity        mean  0.958515                 0.937609   
                          std   0.023576                 0.029169   
average_levenshtein_match mean  0.269649                 0.669730   
                          std   0.149396                 0.285358   
average_similarity_match  mean  0.996461                 0.971218   
                          std   0.003922                 0.013

In [18]:
prompt_mistral_optuna_best = llm_parsers.JSONTuplesPromptTemplate(Path("prompts/optuna_best/best_mistral_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "City", "District", "State", "Region", "Country"]
n_examples = 13
embedding_model = "multi-qa-mpnet-base-dot-v1"
similarity_threshold = 0.5


with FoldEvaluator("Mistral-3-8B-Instruct-model-best", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.MistralAddressParsingModel(
            model_name="mistralai/Ministral-3-8B-Instruct-2512", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_mistral_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            model.example_strategy = embedding_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del embedding_similarity
        del model

Fold 0 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 1 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 2 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 3 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 4 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 5 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 6 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 7 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 8 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 9 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Mistral-3-8B-Instruct-model-best - Metrics for Country City Street House:


accuracy                   mean    0.945606
                           std     0.009917
precision                  mean    0.923552
                           std     0.019655
recall                     mean    0.908730
                           std     0.016812
f1                         mean    0.916052
                           std     0.017427
accuracy_with_tol_1        mean    0.950487
                           std     0.008908
accuracy_with_tol_2        mean    0.953509
                           std     0.007851
accuracy_with_tol_3        mean    0.959558
                           std     0.007312
accuracy_with_tol_4        mean    0.962818
                           std     0.007278
average_levenshtein        mean    0.387431
                           std     0.082008
average_similarity         mean    0.960327
                           std     0.007507
average_levenshtein_match  mean    0.397882
                           std     0.084569
average_similarity_match   mean 

Mistral-3-8B-Instruct-model-best - Specific Metrics:


Field/Entity                    HouseNumber  StreetName      City  District  \
Metric                                                                        
accuracy                  mean     0.962799    0.949836  0.890256  0.933091   
                          std      0.019153    0.020942  0.020734  0.030139   
precision                 mean     0.929573    0.920232  0.915005  0.701005   
                          std      0.038306    0.037513  0.026315  0.160742   
recall                    mean     0.910872    0.899532  0.902360  0.644007   
                          std      0.041329    0.040593  0.019734  0.174436   
f1                        mean     0.920053    0.909443  0.908567  0.659884   
                          std      0.038916    0.034848  0.021541  0.150811   
accuracy_with_tol_1       mean     0.965594    0.952613  0.896764  0.933091   
                          std      0.017019    0.017086  0.021078  0.030139   
accuracy_with_tol_2       mean     0.974888    0.953539  0.898633  0.933091   
                          std      0.013915    0.016267  0.020562  0.030139   
accuracy_with_tol_3       mean     0.986042    0.958186  0.904232  0.934026   
                          std      0.013343    0.014583  0.023371  0.031588   
accuracy_with_tol_4       mean     0.991641    0.960047  0.907953  0.938690   
                          std      0.009211    0.014456  0.021847  0.031467   
average_levenshtein       mean     0.144107    0.478409  0.838880  0.535272   
                          std      0.095749    0.189546  0.198719  0.284813   
average_similarity        mean     0.972349    0.961257  0.921551  0.936161   
                          std      0.013950    0.016632  0.016162  0.029557   
average_levenshtein_match mean     0.146462    0.496356  0.883951  0.575654   
                          std      0.097996    0.202287  0.211680  0.321610   
average_similarity_match  mean     0.984237    0.991673  0.970409  0.994300   
                          std      0.008940    0.006323  0.013374  0.006675   
no_match_rate             mean     0.012089    0.030642  0.050268  0.058550   
                          std      0.009847    0.017462  0.016621  0.026187   

Field/Entity                       State    Region   Country  \
Metric                                                         
accuracy                  mean  0.984181  0.947015  0.979534   
                          std   0.010817  0.023154  0.012226   
precision                 mean  0.895714  0.759178  0.952843   
                          std   0.082435  0.129613  0.031296   
recall                    mean  0.865000  0.772561  0.946353   
                          std   0.136637  0.113231  0.035422   
f1                        mean  0.874038  0.761729  0.949483   
                          std   0.085973  0.105319  0.031673   
accuracy_with_tol_1       mean  0.985107  0.949810  0.986976   
                          std   0.011793  0.020095  0.006519   
accuracy_with_tol_2       mean  0.985107  0.955391  0.986976   
                          std   0.011793  0.016760  0.006519   
accuracy_with_tol_3       mean  0.986976  0.956326  0.989772   
                          std   0.011782  0.018533  0.006852   
accuracy_with_tol_4       mean  0.994410  0.965620  0.991632   
                          std   0.009021  0.017488  0.005263   
average_levenshtein       mean  0.084744  0.349541  0.088326   
                          std   0.097220  0.137865  0.040629   
average_similarity        mean  0.984922  0.951263  0.986152   
                          std   0.011545  0.019169  0.006853   
average_levenshtein_match mean  0.087002  0.368643  0.089452   
                          std   0.101162  0.148194  0.041188   
average_similarity_match  mean  0.999815  0.997589  0.997293   
                          std   0.000586  0.004292  0.003420   
no_match_rate             mean  0.014893  0.046461  0.011163   
                          std   0.011793  0.017399  0.00733

In [19]:
prompt_deepseek_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_deepseek_prompt.txt").read_text())

supported_entities = ['HouseNumber', 'StreetName', 'City', 'District', 'Region', 'Country']
n_examples = 8
embedding_model = "multi-qa-mpnet-base-dot-v1"
similarity_threshold = 0.3


with FoldEvaluator("DeepSeek-R1-Llama-8B-model-best", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.DeepSeekAddressParsingModel(
            model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_deepseek_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 1 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 2 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 3 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 4 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 5 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 6 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 7 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 8 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 9 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
DeepSeek-R1-Llama-8B-model-best - Metrics for Country City Street House:


accuracy                   mean     0.040775
                           std      0.070836
precision                  mean     0.200000
                           std      0.421637
recall                     mean     0.001343
                           std      0.002959
f1                         mean     0.002666
                           std      0.005873
accuracy_with_tol_1        mean     0.041243
                           std      0.070638
accuracy_with_tol_2        mean     0.041474
                           std      0.071089
accuracy_with_tol_3        mean     0.043791
                           std      0.075561
accuracy_with_tol_4        mean     0.048434
                           std      0.078001
average_levenshtein        mean     3.899717
                           std      0.491175
average_similarity         mean     0.041379
                           std      0.071023
average_levenshtein_match  mean    31.478276
                           std     46.565863
average_si

DeepSeek-R1-Llama-8B-model-best - Specific Metrics:


Field/Entity                    HouseNumber  StreetName       City  District  \
Metric                                                                         
accuracy                  mean     0.000000    0.058333   0.031620  0.000000   
                          std      0.000000    0.184466   0.043223  0.000000   
precision                 mean     0.000000    0.000000   0.200000  0.000000   
                          std      0.000000    0.000000   0.421637  0.000000   
recall                    mean     0.000000    0.000000   0.003051  0.000000   
                          std      0.000000    0.000000   0.006875  0.000000   
f1                        mean     0.000000    0.000000   0.006000  0.000000   
                          std      0.000000    0.000000   0.013499  0.000000   
accuracy_with_tol_1       mean     0.000000    0.058333   0.033489  0.000000   
                          std      0.000000    0.184466   0.044339  0.000000   
accuracy_with_tol_2       mean     0.000000    0.058333   0.034415  0.000000   
                          std      0.000000    0.184466   0.045366  0.000000   
accuracy_with_tol_3       mean     0.000000    0.058333   0.037201  0.000000   
                          std      0.000000    0.184466   0.049402  0.000000   
accuracy_with_tol_4       mean     0.000000    0.058333   0.053920  0.000000   
                          std      0.000000    0.184466   0.070977  0.000000   
average_levenshtein       mean     0.942437    4.728003   7.805625  0.933074   
                          std      0.195532    0.697604   0.342306  0.236448   
average_similarity        mean     0.000000    0.058333   0.034028  0.000000   
                          std      0.000000    0.184466   0.044996  0.000000   
average_levenshtein_match mean     0.000000    0.846032  37.682165  0.000000   
                          std      0.000000    2.675387  50.314869  0.000000   
average_similarity_match  mean     0.000000    0.100000   0.394693  0.000000   
                          std      0.000000    0.316228   0.509628  0.000000   
no_match_rate             mean     1.000000    0.941667   0.965585  1.000000   
                          std      0.000000    0.184466   0.045366  0.000000   

Field/Entity                      Region   Country  ApplicantCurrentAddress  \
Metric                                                                        
accuracy                  mean  0.084112  0.073148                 0.020691   
                          std   0.265986  0.231315                 0.046123   
precision                 mean  0.100000  0.000000                 0.000000   
                          std   0.316228  0.000000                 0.000000   
recall                    mean  0.005556  0.000000                 0.000000   
                          std   0.017568  0.000000                 0.000000   
f1                        mean  0.010526  0.000000                 0.000000   
                          std   0.033287  0.000000                 0.000000   
accuracy_with_tol_1       mean  0.084112  0.073148                 0.020691   
                          std   0.265986  0.231315                 0.046123   
accuracy_with_tol_2       mean  0.085981  0.073148                 0.020691   
                          std   0.271897  0.231315                 0.046123   
accuracy_with_tol_3       mean  0.086916  0.079630                 0.024004   
                          std   0.274852  0.251811                 0.054610   
accuracy_with_tol_4       mean  0.091589  0.081481                 0.027559   
                          std   0.289629  0.257667                 0.054880   
average_levenshtein       mean  0.681836  2.122802                 5.805845   
                          std   0.112522  1.561209                 0.209629   
average_similarity        mean  0.084112  0.073157                 0.020691   
                          std   0.265986  0.231341                 0.046123   
average_levenshtein_m

In [20]:
prompt_qwen_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_qwen_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("Qwen3.5-9B-best-from-optuna", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_qwen_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 1 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Qwen3.5-9B-best-from-optuna - Metrics for Country City Street House:


accuracy                   mean    0.955136
                           std     0.011355
precision                  mean    0.927202
                           std     0.017988
recall                     mean    0.930435
                           std     0.014618
f1                         mean    0.928792
                           std     0.015614
accuracy_with_tol_1        mean    0.959320
                           std     0.010956
accuracy_with_tol_2        mean    0.960951
                           std     0.010912
accuracy_with_tol_3        mean    0.966295
                           std     0.008851
accuracy_with_tol_4        mean    0.970020
                           std     0.007619
average_levenshtein        mean    0.312188
                           std     0.077178
average_similarity         mean    0.966598
                           std     0.008885
average_levenshtein_match  mean    0.319164
                           std     0.081673
average_similarity_match   mean 

Qwen3.5-9B-best-from-optuna - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.971166    0.963768      0.914495   
                          std      0.009252    0.017154      0.032549   
precision                 mean     0.932043    0.936583      0.510210   
                          std      0.030627    0.036791      0.125769   
recall                    mean     0.934631    0.925679      0.644505   
                          std      0.022769    0.039026      0.213824   
f1                        mean     0.933210    0.930753      0.566442   
                          std      0.024486    0.033113      0.158919   
accuracy_with_tol_1       mean     0.974888    0.965620      0.916346   
                          std      0.008825    0.015781      0.032038   
accuracy_with_tol_2       mean     0.979543    0.966554      0.919124   
                          std      0.010546    0.016461      0.027959   
accuracy_with_tol_3       mean     0.990689    0.971184      0.919124   
                          std      0.008791    0.014823      0.027959   
accuracy_with_tol_4       mean     0.994418    0.974913      0.925640   
                          std      0.004804    0.014553      0.025016   
average_levenshtein       mean     0.106006    0.336345      0.754898   
                          std      0.054829    0.179742      0.236869   
average_similarity        mean     0.979391    0.973224      0.918064   
                          std      0.006972    0.013392      0.029312   
average_levenshtein_match mean     0.106972    0.343911      0.825190   
                          std      0.055175    0.185975      0.284180   
average_similarity_match  mean     0.988599    0.991636      0.994831   
                          std      0.006032    0.006599      0.004151   
no_match_rate             mean     0.009303    0.018579      0.077146   
                          std      0.006202    0.010684      0.029959   

Field/Entity                        City   Country  ApplicantCurrentAddress  \
Metric                                                                        
accuracy                  mean  0.913525  0.972084                 0.928175   
                          std   0.017896  0.015795                 0.029763   
precision                 mean  0.919659  0.932656                 0.924335   
                          std   0.016783  0.050232                 0.033936   
recall                    mean  0.925086  0.947195                 0.924097   
                          std   0.014302  0.041299                 0.031002   
f1                        mean  0.922332  0.939295                 0.924152   
                          std   0.014483  0.039112                 0.031471   
accuracy_with_tol_1       mean  0.919107  0.977665                 0.933486   
                          std   0.017366  0.012555                 0.027907   
accuracy_with_tol_2       mean  0.919107  0.978600                 0.938014   
                          std   0.017366  0.012431                 0.028337   
accuracy_with_tol_3       mean  0.920976  0.982330                 0.949307   
                          std   0.017977  0.011117                 0.022909   
accuracy_with_tol_4       mean  0.923771  0.986976                 0.957500   
                          std   0.018758  0.008977                 0.021980   
average_levenshtein       mean  0.671452  0.134952                 0.487113   
                          std   0.160774  0.078487                 0.225860   
average_similarity        mean  0.936615  0.977160                 0.947370   
                          std   0.016406  0.012572                 0.022613   
average_levenshtein_match mean  0.695782  0.138670                 0.504163   
                          std   0.178611  0.081064                 0.237382   
average_similarity_match  mean  0.967196  0.998529         

In [21]:
table_label_remapping = {
    # Model names
    "libpostal": "Libpostal",
    "deepparse": "DeepParse",
    "xlm-roberta-large": "XLM-RoBERTa",
    "SpaCy": "SpaCy",
    "Qwen3.5-9B-prompt0-0shot": "Qwen (1st p. 0shot)",
    "Llama-3-8B-model-best": "Llama (Optuna)",
    "Mistral-3-8B-Instruct-model-best": "Mistral (Optuna)",
    "DeepSeek-R1-Llama-8B-model-best": "DeepSeek (Optuna)",
    "Qwen3.5-9B-best-from-optuna": "Qwen (Optuna)",
    # Column names
    "precision": "Precision",
    "recall": "Recall",
    "accuracy": "Accuracy",
    #"f1": "F$_1$-Score" # pandas will escape latex, so do this after generating the table
    "HouseNumber": "House",
    "StreetName": "Street",
    "Neighborhood": "Neigh.",
    "ApplicantCurrentAddress": "App. Addr.",
    "ApplicantBirthPlace": "App. BP",
    # Term "Victim" is deprecated in the code, text uses "Persecutee" 
    "VictimCurrentAddress": "Pers. Addr.",
    "VictimBirthPlace": "Pers. BP",
    "VictimDeathPlace": "Pers. DP"
}

def style_results(df : pd.DataFrame):
    styled_df = df.apply(
        lambda row: pd.Series({
            col : "-" if pd.isna(row[(col, 'mean')]) else f"{row[(col, 'mean')]:.2f}±{row[(col, 'std')]:.2f}"
            for col in row.index.get_level_values(0)
        }, name=row.name),
        axis=1
    )
    styled_df.index.name = "Model"
    styled_df = styled_df.rename(index=lambda l : table_label_remapping.get(l, l)).reset_index()
    styled_df = styled_df.style.apply(
        lambda s: [
            'font-weight: bold' 
            if s.name != "Model" and df[(s.name, 'mean')].iat[k].round(2) == df[(s.name, 'mean')].max().round(2)
            else '' 
            for k in range(len(s))
        ],
        axis=0
    )
    styled_df = styled_df.relabel_index(
        [table_label_remapping.get(l, l) for l in styled_df.columns],
        axis=1
    )
    styled_df = styled_df.apply_index(lambda s: ["font-weight: bold"] * len(s), axis=1)
    styled_df = styled_df.hide(axis="index") # Hide index since we already have a "Model" column
    return styled_df

def print_latex(styled_df) -> str:
    # Extend the LaTeX table to include the standard deviation in the format "mean±std"
    latex_str = styled_df.to_latex(hrules=True, convert_css=True, column_format="l" + "c" * (len(styled_df.columns)))
    latex_str = latex_str.replace(" f1 ", " F$_1$ ")
    latex_str = latex_str.replace("±", "$\\pm$")
    latex_lines = latex_str.splitlines()
    latex_lines[0] += r" % LaTeX table generated in cross_val_evaluation.ipynb"
    latex_str = "\n".join(latex_lines)
    print(latex_str)

In [22]:
all_experiments = pd.concat([
    pd.DataFrame({k : v[["f1", "precision", "recall", "accuracy"]] for k, v in required_results.items()}),
    pd.DataFrame({k : v.unstack(level=1)["f1"] for k, v in specific_results.items()}).loc[["City", "Country", "StreetName"]]
]).T
all_experiments_styled = style_results(all_experiments)

display(all_experiments_styled)

Model,f1,Precision,Recall,Accuracy,City,Country,Street
Libpostal,0.51±0.04,0.64±0.05,0.42±0.04,0.69±0.03,0.40±0.06,0.44±0.11,0.53±0.07
DeepParse,0.29±0.03,0.32±0.03,0.27±0.04,0.57±0.03,0.28±0.05,0.12±0.06,0.06±0.02
XLM-RoBERTa,0.68±0.03,0.71±0.04,0.66±0.03,0.81±0.02,0.56±0.04,0.80±0.07,0.74±0.08
SpaCy,0.83±0.04,0.85±0.03,0.81±0.04,0.89±0.03,0.85±0.05,0.92±0.05,0.77±0.07
Qwen (1st p. 0shot),0.84±0.02,0.86±0.02,0.82±0.02,0.90±0.01,0.83±0.03,0.84±0.05,0.82±0.05
Llama (Optuna),0.91±0.03,0.89±0.03,0.92±0.02,0.94±0.02,0.90±0.02,0.90±0.06,0.91±0.05
Mistral (Optuna),0.92±0.02,0.92±0.02,0.91±0.02,0.95±0.01,0.91±0.02,0.95±0.03,0.91±0.03
DeepSeek (Optuna),0.00±0.01,0.20±0.42,0.00±0.00,0.04±0.07,0.01±0.01,0.00±0.00,0.00±0.00
Qwen (Optuna),0.93±0.02,0.93±0.02,0.93±0.01,0.96±0.01,0.92±0.01,0.94±0.04,0.93±0.03


In [23]:
# Subset of experiments selected for inclusion in the paper.
selected_experiments = all_experiments.drop(index=["DeepSeek-R1-Llama-8B-model-best"])
selected_experiments_styled = style_results(selected_experiments)

display(selected_experiments_styled)

Model,f1,Precision,Recall,Accuracy,City,Country,Street
Libpostal,0.51±0.04,0.64±0.05,0.42±0.04,0.69±0.03,0.40±0.06,0.44±0.11,0.53±0.07
DeepParse,0.29±0.03,0.32±0.03,0.27±0.04,0.57±0.03,0.28±0.05,0.12±0.06,0.06±0.02
XLM-RoBERTa,0.68±0.03,0.71±0.04,0.66±0.03,0.81±0.02,0.56±0.04,0.80±0.07,0.74±0.08
SpaCy,0.83±0.04,0.85±0.03,0.81±0.04,0.89±0.03,0.85±0.05,0.92±0.05,0.77±0.07
Qwen (1st p. 0shot),0.84±0.02,0.86±0.02,0.82±0.02,0.90±0.01,0.83±0.03,0.84±0.05,0.82±0.05
Llama (Optuna),0.91±0.03,0.89±0.03,0.92±0.02,0.94±0.02,0.90±0.02,0.90±0.06,0.91±0.05
Mistral (Optuna),0.92±0.02,0.92±0.02,0.91±0.02,0.95±0.01,0.91±0.02,0.95±0.03,0.91±0.03
Qwen (Optuna),0.93±0.02,0.93±0.02,0.93±0.01,0.96±0.01,0.92±0.01,0.94±0.04,0.93±0.03


In [24]:
print_latex(selected_experiments_styled)

\begin{tabular}{lcccccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries F$_1$ & \bfseries Precision & \bfseries Recall & \bfseries Accuracy & \bfseries City & \bfseries Country & \bfseries Street \\
\midrule
Libpostal & 0.51$\pm$0.04 & 0.64$\pm$0.05 & 0.42$\pm$0.04 & 0.69$\pm$0.03 & 0.40$\pm$0.06 & 0.44$\pm$0.11 & 0.53$\pm$0.07 \\
DeepParse & 0.29$\pm$0.03 & 0.32$\pm$0.03 & 0.27$\pm$0.04 & 0.57$\pm$0.03 & 0.28$\pm$0.05 & 0.12$\pm$0.06 & 0.06$\pm$0.02 \\
XLM-RoBERTa & 0.68$\pm$0.03 & 0.71$\pm$0.04 & 0.66$\pm$0.03 & 0.81$\pm$0.02 & 0.56$\pm$0.04 & 0.80$\pm$0.07 & 0.74$\pm$0.08 \\
SpaCy & 0.83$\pm$0.04 & 0.85$\pm$0.03 & 0.81$\pm$0.04 & 0.89$\pm$0.03 & 0.85$\pm$0.05 & 0.92$\pm$0.05 & 0.77$\pm$0.07 \\
Qwen (1st p. 0shot) & 0.84$\pm$0.02 & 0.86$\pm$0.02 & 0.82$\pm$0.02 & 0.90$\pm$0.01 & 0.83$\pm$0.03 & 0.84$\pm$0.05 & 0.82$\pm$0.05 \\
Llama (Optuna) & 0.91$\pm$0.03 & 0.89$\pm$0.03 & 0.92$\pm$0.02 & 0.94$\pm$0.02 & 0.90$\pm$0.02 & 0.90$\pm$0.06 & 

In [25]:
metric = "f1"
specific_metrics = pd.DataFrame({k : v.unstack(level=1)[metric] for k, v in specific_results.items()}).T


selected_specific_metrics = specific_metrics.drop(columns=["Region", "District"])
# Sanity check that the selected experiments include all the best results
assert all(selected_specific_metrics[c].argmax() == specific_metrics[c].argmax() for c in selected_specific_metrics.columns), "Selected experiments must have the same best metrics as all experiments for the required entities"


entity_specific_metrics = selected_specific_metrics[[c for c in selected_specific_metrics.columns if c[0] in ALL_ENTITIES]]
entity_specific_metrics_styled = style_results(entity_specific_metrics)
display(entity_specific_metrics_styled)
print_latex(entity_specific_metrics_styled)

Model,City,Country,House,Neigh.,State,Street
Libpostal,0.40±0.06,0.44±0.11,0.71±0.05,-,0.54±0.18,0.53±0.07
DeepParse,0.28±0.05,0.12±0.06,0.73±0.07,-,-,0.06±0.02
XLM-RoBERTa,0.56±0.04,0.80±0.07,0.82±0.04,-,0.25±0.17,0.74±0.08
SpaCy,0.85±0.05,0.92±0.05,0.80±0.07,0.38±0.18,0.48±0.20,0.77±0.07
Qwen (1st p. 0shot),0.83±0.03,0.84±0.05,0.88±0.05,0.38±0.15,0.63±0.08,0.82±0.05
Llama (Optuna),0.90±0.02,0.90±0.06,0.92±0.05,-,0.76±0.09,0.91±0.05
Mistral (Optuna),0.91±0.02,0.95±0.03,0.92±0.04,-,0.87±0.09,0.91±0.03
DeepSeek (Optuna),0.01±0.01,0.00±0.00,0.00±0.00,-,-,0.00±0.00
Qwen (Optuna),0.92±0.01,0.94±0.04,0.93±0.02,0.57±0.16,-,0.93±0.03


\begin{tabular}{lccccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries City & \bfseries Country & \bfseries House & \bfseries Neigh. & \bfseries State & \bfseries Street \\
\midrule
Libpostal & 0.40$\pm$0.06 & 0.44$\pm$0.11 & 0.71$\pm$0.05 & - & 0.54$\pm$0.18 & 0.53$\pm$0.07 \\
DeepParse & 0.28$\pm$0.05 & 0.12$\pm$0.06 & 0.73$\pm$0.07 & - & - & 0.06$\pm$0.02 \\
XLM-RoBERTa & 0.56$\pm$0.04 & 0.80$\pm$0.07 & 0.82$\pm$0.04 & - & 0.25$\pm$0.17 & 0.74$\pm$0.08 \\
SpaCy & 0.85$\pm$0.05 & 0.92$\pm$0.05 & 0.80$\pm$0.07 & 0.38$\pm$0.18 & 0.48$\pm$0.20 & 0.77$\pm$0.07 \\
Qwen (1st p. 0shot) & 0.83$\pm$0.03 & 0.84$\pm$0.05 & 0.88$\pm$0.05 & 0.38$\pm$0.15 & 0.63$\pm$0.08 & 0.82$\pm$0.05 \\
Llama (Optuna) & 0.90$\pm$0.02 & 0.90$\pm$0.06 & 0.92$\pm$0.05 & - & 0.76$\pm$0.09 & 0.91$\pm$0.05 \\
Mistral (Optuna) & 0.91$\pm$0.02 & \bfseries 0.95$\pm$0.03 & 0.92$\pm$0.04 & - & \bfseries 0.87$\pm$0.09 & 0.91$\pm$0.03 \\
DeepSeek (Optuna) & 0.01$\pm$0.01 & 0.00$

In [26]:
field_specific_metrics = selected_specific_metrics[[c for c in selected_specific_metrics.columns if c[0] in BZK_ADDRESS_FIELDS]]
field_specific_metrics_styled = style_results(field_specific_metrics)
display(field_specific_metrics_styled)
print_latex(field_specific_metrics_styled)

Model,App. BP,App. Addr.,Pers. BP,Pers. Addr.,Pers. DP
Libpostal,0.53±0.12,0.50±0.07,0.50±0.12,0.51±0.08,0.46±0.24
DeepParse,0.32±0.06,0.26±0.05,0.41±0.09,0.29±0.05,0.32±0.27
XLM-RoBERTa,0.68±0.10,0.68±0.05,0.65±0.07,0.71±0.07,0.64±0.24
SpaCy,0.90±0.04,0.81±0.05,0.94±0.05,0.78±0.06,0.75±0.12
Qwen (1st p. 0shot),0.84±0.08,0.85±0.02,0.82±0.06,0.80±0.07,0.82±0.12
Llama (Optuna),0.93±0.04,0.91±0.04,0.94±0.04,0.87±0.07,0.84±0.19
Mistral (Optuna),0.93±0.04,0.92±0.03,0.94±0.05,0.88±0.06,0.86±0.10
DeepSeek (Optuna),0.01±0.02,0.00±0.00,0.02±0.05,0.00±0.00,0.02±0.07
Qwen (Optuna),0.94±0.03,0.92±0.03,0.96±0.04,0.91±0.03,0.94±0.11


\begin{tabular}{lcccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries App. BP & \bfseries App. Addr. & \bfseries Pers. BP & \bfseries Pers. Addr. & \bfseries Pers. DP \\
\midrule
Libpostal & 0.53$\pm$0.12 & 0.50$\pm$0.07 & 0.50$\pm$0.12 & 0.51$\pm$0.08 & 0.46$\pm$0.24 \\
DeepParse & 0.32$\pm$0.06 & 0.26$\pm$0.05 & 0.41$\pm$0.09 & 0.29$\pm$0.05 & 0.32$\pm$0.27 \\
XLM-RoBERTa & 0.68$\pm$0.10 & 0.68$\pm$0.05 & 0.65$\pm$0.07 & 0.71$\pm$0.07 & 0.64$\pm$0.24 \\
SpaCy & 0.90$\pm$0.04 & 0.81$\pm$0.05 & 0.94$\pm$0.05 & 0.78$\pm$0.06 & 0.75$\pm$0.12 \\
Qwen (1st p. 0shot) & 0.84$\pm$0.08 & 0.85$\pm$0.02 & 0.82$\pm$0.06 & 0.80$\pm$0.07 & 0.82$\pm$0.12 \\
Llama (Optuna) & 0.93$\pm$0.04 & 0.91$\pm$0.04 & 0.94$\pm$0.04 & 0.87$\pm$0.07 & 0.84$\pm$0.19 \\
Mistral (Optuna) & 0.93$\pm$0.04 & \bfseries 0.92$\pm$0.03 & 0.94$\pm$0.05 & 0.88$\pm$0.06 & 0.86$\pm$0.10 \\
DeepSeek (Optuna) & 0.01$\pm$0.02 & 0.00$\pm$0.00 & 0.02$\pm$0.05 & 0.00$\pm$0.00 & 0.02$\pm$0

# Qwen Specific tables

In [27]:
qwen_specific_results = (
    specific_results["Qwen3.5-9B-best-from-optuna"].unstack(level=[1, 2])[["f1", "precision", "recall", "accuracy"]]
).T

qwen_entity_results = pd.concat([
    qwen_specific_results[["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]],
    required_results["Qwen3.5-9B-best-from-optuna"][["f1", "precision", "recall", "accuracy"]]
], axis=1).T
qwen_entity_results.rename(index={0: "House, Street, City, Country"}, inplace=True)

qwen_entity_results_styled = style_results(qwen_entity_results)
display(qwen_entity_results_styled)
print_latex(qwen_entity_results_styled)


Model,f1,Precision,Recall,Accuracy
House,0.93±0.02,0.93±0.03,0.93±0.02,0.97±0.01
Street,0.93±0.03,0.94±0.04,0.93±0.04,0.96±0.02
Neigh.,0.57±0.16,0.51±0.13,0.64±0.21,0.91±0.03
City,0.92±0.01,0.92±0.02,0.93±0.01,0.91±0.02
Country,0.94±0.04,0.93±0.05,0.95±0.04,0.97±0.02
"House, Street, City, Country",0.93±0.02,0.93±0.02,0.93±0.01,0.96±0.01


\begin{tabular}{lccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries F$_1$ & \bfseries Precision & \bfseries Recall & \bfseries Accuracy \\
\midrule
House & 0.93$\pm$0.02 & 0.93$\pm$0.03 & 0.93$\pm$0.02 & \bfseries 0.97$\pm$0.01 \\
Street & 0.93$\pm$0.03 & \bfseries 0.94$\pm$0.04 & 0.93$\pm$0.04 & 0.96$\pm$0.02 \\
Neigh. & 0.57$\pm$0.16 & 0.51$\pm$0.13 & 0.64$\pm$0.21 & 0.91$\pm$0.03 \\
City & 0.92$\pm$0.01 & 0.92$\pm$0.02 & 0.93$\pm$0.01 & 0.91$\pm$0.02 \\
Country & \bfseries 0.94$\pm$0.04 & 0.93$\pm$0.05 & \bfseries 0.95$\pm$0.04 & \bfseries 0.97$\pm$0.02 \\
House, Street, City, Country & 0.93$\pm$0.02 & 0.93$\pm$0.02 & 0.93$\pm$0.01 & 0.96$\pm$0.01 \\
\bottomrule
\end{tabular}


In [29]:
qwen_bzk_field_styled = style_results(qwen_specific_results[[f for f in BZK_ADDRESS_FIELDS]].T.sort_index())
display(qwen_bzk_field_styled)
print_latex(qwen_bzk_field_styled)

Model,f1,Precision,Recall,Accuracy
App. BP,0.94±0.03,0.93±0.04,0.94±0.04,0.97±0.01
App. Addr.,0.92±0.03,0.92±0.03,0.92±0.03,0.93±0.03
Pers. BP,0.96±0.04,0.96±0.03,0.97±0.06,0.99±0.02
Pers. Addr.,0.91±0.03,0.91±0.03,0.91±0.03,0.93±0.03
Pers. DP,0.94±0.11,0.92±0.16,0.98±0.05,0.98±0.03


\begin{tabular}{lccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries F$_1$ & \bfseries Precision & \bfseries Recall & \bfseries Accuracy \\
\midrule
App. BP & 0.94$\pm$0.03 & 0.93$\pm$0.04 & 0.94$\pm$0.04 & 0.97$\pm$0.01 \\
App. Addr. & 0.92$\pm$0.03 & 0.92$\pm$0.03 & 0.92$\pm$0.03 & 0.93$\pm$0.03 \\
Pers. BP & \bfseries 0.96$\pm$0.04 & \bfseries 0.96$\pm$0.03 & 0.97$\pm$0.06 & \bfseries 0.99$\pm$0.02 \\
Pers. Addr. & 0.91$\pm$0.03 & 0.91$\pm$0.03 & 0.91$\pm$0.03 & 0.93$\pm$0.03 \\
Pers. DP & 0.94$\pm$0.11 & 0.92$\pm$0.16 & \bfseries 0.98$\pm$0.05 & 0.98$\pm$0.03 \\
\bottomrule
\end{tabular}
